# Évaluation CL — EWC Online + MLP — Dataset 3 PRONOSTIA — by_condition

| Champ | Valeur |
|-------|--------|
| **Modèle** | EWC Online + MLP (993 paramètres, input_dim=13, hidden=[32, 16]) |
| **Dataset** | FEMTO PRONOSTIA — roulements à billes industriels |
| **Scénario** | by_condition : Condition 1 → Condition 2 → Condition 3 (3 tâches) |
| **Expérience** | exp_050 — voir experiments/exp_050_ewc_pronostia_by_condition/config_snapshot.yaml |
| **Sprint** | 10 — S10-06 |

> **Modèle supervisé** : EWC reçoit les labels à l'entraînement.  
> Sortie = probabilité de défaut ŷ ∈ [0, 1] (sigmoid).  
> RAM = 1 171 B — meilleure empreinte mémoire parmi les modèles supervisés.  
> Stratégie CL : régularisation EWC Online (λ=1000, γ=0.9) — aucun buffer de replay.  
> **Gap 1** : premier résultat CL sur données industrielles réelles de roulements (PRONOSTIA).

```bash
jupyter nbconvert --to notebook --execute \
    notebooks/cl_eval/pronostia_by_condition/ewc.ipynb \
    --output /tmp/ewc_pronostia_executed.ipynb --ExecutePreprocessor.timeout=600
```

In [ ]:
# Section 1 — Setup & imports
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim
from IPython.display import Image, Markdown, display

# --- CWD navigation : notebook 3 niveaux de profondeur ---
_cwd = Path(".").resolve()
if _cwd.name == "pronostia_by_condition":
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == "cl_eval":
    os.chdir(_cwd.parent.parent)
elif _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.plots import (
    plot_accuracy_matrix,
    plot_confusion_matrix_grid,
    plot_forgetting_curve,
    plot_roc_curves_per_task,
    save_figure,
)
from src.evaluation.feature_space_plots import (
    fit_pca2d,
    plot_feature_space_2d,
)

# --- Chemins ---
EXP_DIR     = REPO_ROOT / "experiments/exp_050_ewc_pronostia_by_condition/results"
FIGURES_DIR = REPO_ROOT / "notebooks/figures/cl_evaluation/ewc/pronostia/by_condition"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

NPY_DIR         = REPO_ROOT / "data/raw/Pronostia dataset/binaries"
NORMALIZER_PATH = REPO_ROOT / "configs/pronostia_normalizer.yaml"

# --- Constantes ---
TASK_NAMES    = ["Condition 1 (1800rpm, 4000N)", "Condition 2 (1650rpm, 4200N)", "Condition 3 (1500rpm, 5000N)"]
MODEL_NAME    = "EWC"
DATA_AVAILABLE = NPY_DIR.exists() and NORMALIZER_PATH.exists()

print(f"REPO_ROOT       : {REPO_ROOT}")
print(f"EXP_DIR         : {EXP_DIR}")
print(f"FIGURES_DIR     : {FIGURES_DIR}")
print(f"NPY disponible  : {DATA_AVAILABLE}")
print(f"Date exécution  : {datetime.now():%Y-%m-%d %H:%M}")

if not DATA_AVAILABLE:
    display(Markdown(
        "> ⚠️ **NPY absent** — Sections 5, 6, 7, 8 en mode dégradé (données synthétiques). "
        "Placer les fichiers `data/raw/Pronostia dataset/binaries/` pour le mode complet."
    ))

In [ ]:
# Section 2 — Chargement des résultats exp_050

metrics_path    = EXP_DIR / "metrics.json"
acc_matrix_path = EXP_DIR / "acc_matrix_ewc.npy"

metrics    = json.loads(metrics_path.read_text())
acc_matrix = np.load(acc_matrix_path, allow_pickle=True)

# Extraire les métriques EWC depuis la structure imbriquée
cl  = metrics["cl_metrics"]["ewc"]
mem = metrics["cl_metrics"]["memory"]["forward"]

# Reconstruire la matrice acc numpy (remplacement null → NaN)
acc_matrix_json = np.array(
    [[v if v is not None else np.nan for v in row] for row in cl["acc_matrix"]],
    dtype=float,
)

print("=" * 55)
print(f"  Modèle         : EWC Online + MLP")
print(f"  AA             = {cl['aa']:.4f}")
print(f"  AF             = {cl['af']:.4f}")
print(f"  BWT            = {cl['bwt']:+.4f}")
print(f"  FWT            = {cl['fwt']:.4f}")
print(f"  Forgetting/tâche: {[round(v, 4) for v in cl['forgetting_per_task']]}")
print(f"  RAM peak       = {mem['ram_peak_bytes']} B ({mem['ram_peak_bytes']/1024:.2f} Ko)")
print(f"  Latence        = {mem['inference_latency_ms']:.5f} ms")
print(f"  n_params       = {mem['n_params']}")
print(f"  Budget 64 Ko   : {mem['within_budget_64ko']}")
print("=" * 55)
print("\nMatrice acc (3×3) :")
print(acc_matrix_json)

In [ ]:
# Section 3 — Matrice d'accuracy (heatmap)
# acc_matrix[i, j] = accuracy sur tâche j après entraînement sur tâche i
# Triangle supérieur = NaN (tâche pas encore vue)

fig = plot_accuracy_matrix(
    acc_matrix_json,
    task_names=TASK_NAMES,
    title=f"{MODEL_NAME} — pronostia/by_condition",
)
save_figure(fig, FIGURES_DIR / "acc_matrix.png")
display(Image(str(FIGURES_DIR / "acc_matrix.png")))

In [ ]:
# Section 4 — Courbe d'oubli par tâche
# AF = 0.000 → courbe plate : EWC bien calibré (λ=1000) sur PRONOSTIA

fig = plot_forgetting_curve(
    acc_matrix_json,
    task_names=TASK_NAMES,
    title=f"{MODEL_NAME} — Évolution accuracy par tâche (pronostia/by_condition)",
)
save_figure(fig, FIGURES_DIR / "forgetting_curve.png")
display(Image(str(FIGURES_DIR / "forgetting_curve.png")))

In [ ]:
# Section 5 — Rejeu du scénario CL (collecte preds_dict + proba_dict)
# Reproduit exactement la config exp_050 : seed=42, epochs=10/task, SGD lr=0.01,
# λ=1000, γ=0.9, n_fisher_samples=200, hidden=[32,16], dropout=0.2, input_dim=13

preds_dict  = {}  # (i, j) → (y_true, y_pred_binary)
proba_dict  = {}  # (i, j) → sigmoid outputs float32 (pour ROC)
X_tests_raw = []  # [N_val, 13] par tâche — pour la viz PCA
y_tests_raw = []  # [N_val] par tâche

if DATA_AVAILABLE:
    from src.data.pronostia_dataset import get_pronostia_dataloaders
    from src.models.ewc import EWCMlpClassifier
    from src.models.ewc.fisher import compute_fisher_diagonal, update_fisher_online
    from src.utils.reproducibility import set_seed

    set_seed(42)

    tasks = get_pronostia_dataloaders(
        npy_dir=NPY_DIR,
        normalizer_path=NORMALIZER_PATH,
        batch_size=32,
        seed=42,
    )

    # Extraire les données de validation en numpy une seule fois
    for t in tasks:
        X_v = np.concatenate([b[0].numpy() for b in t["val_loader"]])
        y_v = np.concatenate([b[1].numpy().flatten() for b in t["val_loader"]])
        X_tests_raw.append(X_v)
        y_tests_raw.append(y_v)

    # Instancier le modèle EWC (même config que exp_050 — input_dim=13 pour PRONOSTIA)
    model = EWCMlpClassifier(input_dim=13, hidden_dims=[32, 16], dropout=0.2)
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

    fisher     = None
    theta_star = None
    EWC_LAMBDA  = 1000
    EWC_GAMMA   = 0.9
    N_FISHER    = 200
    EPOCHS      = 10

    for i, task in enumerate(tasks):
        domain = task.get("domain", f"Condition {task['task_id']}")
        print(f"\n--- Tâche {i + 1}/3 : {domain} ---")

        model.train()
        for epoch in range(EPOCHS):
            losses = []
            for x, y in task["train_loader"]:
                optimizer.zero_grad()
                loss = model.ewc_loss(x, y, fisher, theta_star, EWC_LAMBDA)
                loss.backward()
                optimizer.step()
                losses.append(loss.item())
            if (epoch + 1) % 5 == 0:
                print(f"  Epoch {epoch+1:2d}/{EPOCHS} | Loss: {np.mean(losses):.4f}")

        theta_star = model.get_theta_star()
        new_fisher = compute_fisher_diagonal(model, task["train_loader"], "cpu", n_samples=N_FISHER)
        fisher = update_fisher_online(fisher, new_fisher, gamma=EWC_GAMMA)
        print(f"  Fisher mis à jour (γ={EWC_GAMMA})")

        model.eval()
        with torch.no_grad():
            for j in range(i + 1):
                X_j = torch.from_numpy(X_tests_raw[j]).float()
                probas = model(X_j).numpy().flatten()
                y_pred = (probas >= 0.5).astype(float)
                preds_dict[(i, j)]  = (y_tests_raw[j], y_pred)
                proba_dict[(i, j)]  = probas.astype(np.float32)
                acc = (y_tests_raw[j] == y_pred).mean()
                print(f"  preds_dict[({i},{j})] → N={len(y_tests_raw[j])}, acc={acc:.4f}")

    print(f"\nScénario CL rejoué — {len(preds_dict)} évaluations collectées")

else:
    display(Markdown("> ⚠️ **Mode dégradé** — NPY absent. preds_dict synthétique depuis acc_matrix."))

    N_SYNTH = 500
    rng = np.random.default_rng(42)
    # PRONOSTIA : ~10% positifs (déséquilibré)
    y_synth = np.concatenate([np.zeros(int(N_SYNTH * 0.9)), np.ones(int(N_SYNTH * 0.1))])

    for i in range(3):
        for j in range(i + 1):
            noise = rng.normal(0, 0.1, len(y_synth))
            probas_synth = np.where(y_synth == 1, 0.75 + noise, 0.25 + noise).clip(0, 1)
            y_pred_synth = (probas_synth >= 0.5).astype(float)
            preds_dict[(i, j)]  = (y_synth.copy(), y_pred_synth)
            proba_dict[(i, j)]  = probas_synth.astype(np.float32)

    print("preds_dict synthétique créé (mode dégradé — NPY absent)")

In [ ]:
# Section 6 — Matrices de confusion par tâche (grille 3×3)
# Normalisées par ligne (recall par classe)

fig = plot_confusion_matrix_grid(
    preds_dict,
    task_names=TASK_NAMES,
    model_name=MODEL_NAME,
    threshold=0.5,
)
save_figure(fig, FIGURES_DIR / "confusion_matrix_grid.png")
display(Image(str(FIGURES_DIR / "confusion_matrix_grid.png")))

In [ ]:
# Section 7 — Courbes ROC par tâche
# PRONOSTIA : ~10% positifs → AUC-ROC plus pertinent qu'accuracy seule

cl_loaded = metrics["cl_metrics"]["ewc"]
aa  = cl_loaded["aa"]
af  = cl_loaded["af"]
bwt = cl_loaded["bwt"]

fig = plot_roc_curves_per_task(
    preds_dict,
    scores_dict=proba_dict,
    task_names=TASK_NAMES,
    model_name=MODEL_NAME,
)

fig.text(
    0.5, 0.01,
    f"AA exp_050 = {aa:.4f}  |  AF = {af:.4f}  |  BWT = {bwt:+.4f}",
    ha="center", fontsize=9, color="#444444",
)

save_figure(fig, FIGURES_DIR / "roc_curves.png")
display(Image(str(FIGURES_DIR / "roc_curves.png")))

In [ ]:
# Section 8 — Espace des features (PCA 2D)
# Coloré par condition (1/2/3) et par label (normal/faulty)
# Montre la séparabilité inter-conditions dans l'espace features appris

if DATA_AVAILABLE and len(X_tests_raw) == 3:
    X_all      = np.concatenate(X_tests_raw, axis=0)   # [N_total, 13]
    y_all      = np.concatenate(y_tests_raw, axis=0)   # [N_total]
    domain_ids = np.concatenate([
        np.full(len(X_tests_raw[k]), k) for k in range(3)
    ])

    pca, X_proj = fit_pca2d(X_all)
    expl_var = pca.explained_variance_ratio_
    xlabel = f"PC1 ({expl_var[0]*100:.1f}%)"
    ylabel = f"PC2 ({expl_var[1]*100:.1f}%)"

    fig, ax = plt.subplots(figsize=(9, 7))

    plot_feature_space_2d(
        X_proj, y_all,
        title=f"{MODEL_NAME} — Espace features PCA 2D — pronostia/by_condition",
        ax=ax,
        domain_ids=domain_ids,
        alpha=0.25,
        s=6,
        xlabel=xlabel,
        ylabel=ylabel,
    )

    TASK_COLORS = ["#2196F3", "#FF9800", "#9C27B0"]
    for k, (name, color) in enumerate(zip(TASK_NAMES, TASK_COLORS)):
        mask = domain_ids == k
        cx, cy = X_proj[mask, 0].mean(), X_proj[mask, 1].mean()
        ax.annotate(
            f"C{k+1}",
            xy=(cx, cy),
            fontsize=11,
            fontweight="bold",
            color=color,
            ha="center",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
        )

    fig.tight_layout()
    save_figure(fig, FIGURES_DIR / "feature_space_pca.png")
    display(Image(str(FIGURES_DIR / "feature_space_pca.png")))

else:
    display(Markdown(
        "> ⚠️ **Section 8 ignorée** — NPY absent ou scénario CL non rejoué. "
        "feature_space_pca.png non généré."
    ))
    print("[SKIP] feature_space_pca.png — données non disponibles.")

## Discussion — Gap 1

Ces résultats sur FEMTO PRONOSTIA (données industrielles réelles de roulements à billes) complètent
les validations sur les datasets Kaggle (Equipment Monitoring, Pump Maintenance).

**Comparaison cross-dataset** :
- Dataset Monitoring (by_equipment, 3 tâches) : AA ≈ 0.9824, AF ≈ 0.0010
- Dataset Pump (by_pump_id) : AA = ?, AF = ?
- **Dataset PRONOSTIA (by_condition, 3 tâches) : AA = 0.9819, AF = 0.0000** ← exp_050

**Particularités PRONOSTIA vs. Monitoring** :
- Classes déséquilibrées (~10% positifs) → AUC-ROC est la métrique principale
- 13 features (vs. 4) : fenêtrage temporel sur signaux d'accélération
- Drift = conditions opératoires (vitesse + charge) plutôt que type d'équipement

`FIXME(gap1)` : ✅ Premier résultat CL sur données industrielles réelles de roulements —
voir `docs/roadmap_phase1.md` section Sprint 10 pour la synthèse complète.

`TODO(arnaud)` : La section Discussion Gap 1 doit-elle inclure une référence bibliographique
au protocole IEEE PHM 2012 Challenge (`@PHM2012`) pour contextualiser dans la littérature ?

In [ ]:
# Section 10 — Tableau récapitulatif + critères d'acceptation

cl_final = metrics["cl_metrics"]["ewc"]
mem_final = metrics["cl_metrics"]["memory"]["forward"]

aa    = cl_final["aa"]
af    = cl_final["af"]
bwt   = cl_final["bwt"]
ram_b = mem_final["ram_peak_bytes"]
ram_ko = ram_b / 1024
lat   = mem_final["inference_latency_ms"]
n_par = mem_final["n_params"]
forgetting_per_task = cl_final.get("forgetting_per_task", [])

display(Markdown("### Résultats finaux — EWC Online + MLP — pronostia/by_condition (exp_050)"))

recap_table = f"""
| Modèle | AA ↑ | AF ↓ | BWT | RAM ↓ | Latence ↓ | n_params |
|--------|------|------|-----|-------|-----------|----------|
| {MODEL_NAME} | {aa:.4f} | {af:.4f} | {bwt:+.4f} | {ram_ko:.2f} Ko | {lat:.5f} ms | {n_par} |
"""
display(Markdown(recap_table))

print(f"Forgetting par tâche (C1, C2) : {[round(v, 4) for v in forgetting_per_task]}")
print()
print("=" * 60)
print("  NOTE SCIENTIFIQUE — Gap 2 (contrainte embarquée STM32N6)")
print("=" * 60)
print(f"  RAM = {ram_b} B = {ram_ko:.2f} Ko")
print(f"  Budget STM32N6 : 65 536 B (64 Ko)")
print(f"  Marge disponible : {65536 - ram_b} B ({(65536 - ram_b)/1024:.1f} Ko)")
print(f"  EWC occupe {ram_b / 65536 * 100:.2f}% du budget RAM")
print(f"  État EWC (Fisher + θ*) = 2 × {n_par} × 4B = {2 * n_par * 4} B supplémentaires")
print()

# Vérification des critères d'acceptation (S10-06)
print("=" * 60)
print("  Critères d'acceptation (S10-06)")
print("=" * 60)
for fig_name in ["acc_matrix.png", "forgetting_curve.png", "confusion_matrix_grid.png",
                 "roc_curves.png", "feature_space_pca.png"]:
    status = "OK" if (FIGURES_DIR / fig_name).exists() else "MANQUANTE"
    print(f"  [{status}] {fig_name}")

print()
print(f"  [{'OK' if abs(aa - 0.9819) < 0.005 else 'WARN'}] AA     = {aa:.4f}  (attendu ≈ 0.9819)")
print(f"  [{'OK' if af < 0.005 else 'WARN'}] AF     = {af:.4f}  (attendu ≈ 0.0000)")
print(f"  [{'OK' if abs(bwt) < 0.01 else 'WARN'}] BWT    = {bwt:+.4f} (attendu ≈ +0.0049)")
print(f"  [{'OK' if ram_b <= 65536 else 'FAIL'}] RAM    = {ram_ko:.2f} Ko (contrainte ≤ 64 Ko)")
print(f"  [{'OK' if lat < 100.0 else 'WARN'}] Latence= {lat:.5f} ms (contrainte ≤ 100 ms)")